# 02 — Hugging Face Transformers Basics

**Goal**: Load real transformer models and see the architecture in action.

We'll use small, free models from HuggingFace to explore tokenization, attention, and generation.

In [ ]:
# !pip install transformers torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import numpy as np

print("Imports OK")

## 1. Tokenizers — See How LLMs See Your Prompts

Every model has its own tokenizer. Let's see what happens when you type a prompt.

In [ ]:
# Load a small tokenizer (GPT-2 tokenizer — similar to what Llama uses)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

prompt = "I love building AI applications"
tokens = tokenizer(prompt)

print(f"Prompt: {prompt}")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Decoded tokens: {[tokenizer.decode([t]) for t in tokens['input_ids']]}")
print(f"Number of tokens: {len(tokens['input_ids'])}")

In [ ]:
# Compare different prompts
prompts = [
    "Hello, how are you?",
    "The quick brown fox jumps over the lazy dog",
    "```python\nprint('hello')\n```",  # code
    "http://example.com/api/v1/users?id=123"  # URL
]

for p in prompts:
    tokens = tokenizer(p)
    decoded = [tokenizer.decode([t]) for t in tokens['input_ids']]
    print(f"\n{p[:50]}...")
    print(f"  Tokens: {len(tokens['input_ids'])} → {decoded[:8]}...")

### Key Insight: Tokenization Is NOT Simple Words

Notice:
- "http://example.com" gets split into subwords: `['http', '://', 'example', '.com']`
- Code tokens (`print`, `hello`) get separate tokens
- **Token count ≠ word count** — this matters for context windows!
- A 4K context window ≈ ~3,000 English words
- A 128K context window ≈ ~96,000 words

## 2. Load a Tiny Generation Model

We'll use **GPT-2** (124M params) — small enough to run on any laptop.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
print(f"Model loaded! Parameters: {model.num_parameters() / 1e6:.0f}M")

## 3. Generate Text

Let's see how the model predicts one token at a time.

In [ ]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

result = generator(
    "The future of artificial intelligence is",
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7
)

print(result[0]["generated_text"])

## 4. Temperature — The Creativity Dial

Temperature controls randomness in token selection.

- **Low (0.1)**: Always picks the most likely token (deterministic, repetitive)
- **Medium (0.7)**: Balanced creativity
- **High (1.5)**: Very random, may produce nonsense

In [ ]:
prompt = "The best thing about machine learning is"

for temp in [0.1, 0.7, 1.5]:
    result = generator(
        prompt,
        max_new_tokens=25,
        do_sample=True,
        temperature=temp
    )
    print(f"\nTemperature {temp}:")
    print(f"  {result[0]['generated_text']}")

## 5. Top-p / Top-k Sampling

Alternative controls:
- **Top-k**: Only sample from the top K most likely tokens
- **Top-p**: Sample from the smallest set of tokens whose probability sums to P

In [ ]:
prompt = "Once upon a time in a magical kingdom"

# Greedy (always pick best token)
result = generator(prompt, max_new_tokens=25, do_sample=False)
print(f"Greedy:    {result[0]['generated_text']}")

# Top-k (sample from top 50)
result = generator(prompt, max_new_tokens=25, do_sample=True, top_k=50, temperature=0.8)
print(f"Top-k=50:  {result[0]['generated_text']}")

# Top-p (nucleus sampling)
result = generator(prompt, max_new_tokens=25, do_sample=True, top_p=0.9, temperature=0.8)
print(f"Top-p=0.9: {result[0]['generated_text']}")

## 6. Understanding the Transformer Inference Loop

Let's manually step through generation to see what happens inside.

In [ ]:
def manual_generate(prompt, max_tokens=5):
    """Generate one token at a time, showing probabilities."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    print(f"Starting with: '{prompt}'")
    print(f"Input tokens: {input_ids.shape[1]}")
    
    for step in range(max_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
            
        # Get logits for the last token
        next_token_logits = outputs.logits[0, -1, :]
        
        # Convert to probabilities
        probs = torch.softmax(next_token_logits, dim=-1)
        
        # Get top 5 candidates
        top5_probs, top5_indices = torch.topk(probs, 5)
        
        print(f"\nStep {step + 1} — Top 5 candidates:")
        for p, idx in zip(top5_probs, top5_indices):
            token = tokenizer.decode([idx])
            print(f"  '{token}': {p:.3%}")
        
        # Pick the most likely
        next_token = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)
        input_ids = torch.cat([input_ids, next_token], dim=-1)
    
    print(f"\nFinal output: {tokenizer.decode(input_ids[0])}")

manual_generate("I think AI will", max_tokens=3)

## Key Takeaways

1. **Tokenizers** split text into subword tokens — not words
2. **Generation** is iterative: predict token → feed back → repeat
3. **Temperature**: controls randomness (low = predictable, high = creative)
4. **Top-k/Top-p**: additional sampling strategies
5. The model outputs **probabilities** for every token in its vocabulary
6. GPT-2 (124M) is tiny compared to modern LLMs (7B-70B)!

**Next**: Use your local Ollama models for inference →